# LIGO/Virgo CBC Detection Pipeline: From Population to Sky Map

This notebook walks through the full gravitational wave detection simulation
pipeline step by step, reproducing what the `Makefile` does automatically --
but interactively and with full visibility into each stage:

1. **Download** waveform data from Zenodo
2. **Load** detector PSDs and population samples (`bgp.h5` / `fullpop4.h5`)
3. **Inject** simulated signals (`bayestar-inject`)
4. **Detect** coincidences (`bayestar-realize-coincs`)
5. **Localize** detected events (`bayestar-localize-coincs`)

## Imports

In [ ]:
import copy
import logging
import os
import warnings

import igwn_segments as segments
import lal
import lal.series
import lalsimulation
import numpy as np
from astropy import cosmology, units
from astropy.table import Table
from igwn_ligolw import ligolw, lsctables
from igwn_ligolw import utils as ligolw_utils
from igwn_ligolw.ligolw import Param
from igwn_ligolw.lsctables import (
    CoincDefTable,
    CoincInspiralTable,
    CoincMapTable,
    CoincTable,
    ProcessParamsTable,
    ProcessTable,
    SimInspiralTable,
    SnglInspiralTable,
    TimeSlideTable,
)
from igwn_ligolw.utils import process as ligolw_process
from igwn_ligolw.utils.search_summary import append_search_summary
from lalinspiral.inspinjfind import InspiralSCExactCoincDef
from lalinspiral.thinca import InspiralCoincDef
from ligo.skymap.bayestar import filter as bayestar_filter
from ligo.skymap.io.events.ligolw import ContentHandler
from ligo.skymap.tool import write_fileobj
from ligo.skymap.tool.bayestar_inject import GWCosmo
from ligo.skymap.tool.bayestar_realize_coincs import simulate
from ligo.skymap.util import progress_map
from scipy import stats
from scipy.interpolate import interp1d

# Suppress BAYESTAR PSD warnings
logging.getLogger("BAYESTAR").setLevel(logging.ERROR)

# Suppress all Python warnings
warnings.filterwarnings("ignore")

print("All imports successful!")

All imports successful!


In [78]:
# Suppress ligo.skymap PSD warnings
logging.getLogger("BAYESTAR").setLevel(logging.ERROR)

# Suppress all Python warnings
warnings.filterwarnings("ignore")

# Step 1: Download the Waveform Data
### LALSuite >= 7.25 requires data files hosted on Zenodo:

In [79]:
# Set the Environment Variable (before any lal import!)

os.environ["LAL_DATA_PATH"] = os.path.join(os.getcwd(), "lalsuite-waveform-data")

In [80]:
%%bash
mkdir -p lalsuite-waveform-data

# Check if already downloaded to avoid re-downloading
if [ ! -f lalsuite-waveform-data/SEOBNRv4ROM_v3.0.hdf5 ]; then
    echo "Downloading SEOBNRv4ROM data..."
    curl -L https://zenodo.org/records/14999310/files/SEOBNRv4ROM_v3.0.hdf5 \
        -o lalsuite-waveform-data/SEOBNRv4ROM_v3.0.hdf5
    echo "Done!"
else
    echo "Already downloaded, skipping."
fi

Already downloaded, skipping.


In [81]:
%%bash
ls -lh lalsuite-waveform-data/

total 655800
-rw-r--r--@ 1 rk281962  2048330233   318M May 14 11:02 SEOBNRv4ROM_v3.0.hdf5


# Step 2: Load the PSDs

In [82]:
with open("runs/O4HL/psds.xml", "rb") as f:
    psds = list(
        lal.series.read_psd_xmldoc(
            ligolw_utils.load_fileobj(f, contenthandler=lal.series.PSDContentHandler)
        ).values()
    )

print(f"Number of detectors: {len(psds)}")
for p in psds:
    print(f"  {p.name}: f0={p.f0} Hz, df={p.deltaF} Hz, N={len(p.data.data)}")

Number of detectors: 2
  o4b_h1_ref: f0=10.0 Hz, df=1.0 Hz, N=4086
  o4b_l1_ref: f0=10.0 Hz, df=1.0 Hz, N=4086


# Step 3: Load samples and take only the first few ones

In [83]:
samples = Table.read("fullpop4.h5")

print(f"Total samples: {len(samples)}")
print(samples[:4])

Total samples: 1165080
      mass1              mass2               spin1z               spin2z       
------------------ ------------------ -------------------- --------------------
24.097117511166395  18.15663201544273   0.1042061773132673  0.14050206117655498
  1.66749780975944 1.2257760890171845 0.044977863704632144  0.25706824711600046
2.0126068705996163  1.129031649093662  0.05222793257066645  0.14805602335454707
 1.967645205156753 1.1647068598207864  0.05870975456708861 0.045146180087644636


### Build GWCosmo and run on first few events

Before running, we need to understand the `waveform` parameter.

`"o2-uberbank"` is not a waveform itself -- it is a **selection rule** from the
ER10/O2 uberbank that automatically picks the best waveform based on the total
mass of the system, as defined in `ligo/skymap/bayestar/filter.py`:

| Total mass | Waveform chosen | Notes |
|---|---|---|
| < 4 Msun | `TaylorF2threePointFivePN` | BNS, fast analytical |
| >= 4 Msun | `SEOBNRv4_ROM` | NSBH/BBH, needs HDF5 data file |

This is why downloading `SEOBNRv4ROM_v3.0.hdf5` from Zenodo is mandatory --
without it, every system with total mass >= 4 Msun (BBH, NSBH, and even
heavy BNS) silently falls back to `TaylorF2`,
which is **physically incorrect** for high-mass systems and will produce
wrong horizon distances.

Now we can safely build `GWCosmo` and compute the maximum sensitive distance
for the first few events as a sanity check before launching the full run:

In [84]:
# test_samples = samples[:1000]

# Select 1000 random samples from the full population
rng = np.random.default_rng(seed=0)  # reproducible
indices = rng.choice(len(samples), size=1000, replace=False)
test_samples = samples[indices]

# test_samples = samples[np.random.choice(len(samples), size=1000, replace=False)]

print(f"Total samples:  {len(samples)}")
print(f"Selected:       {len(test_samples)}")
print(test_samples[:4])

Total samples:  1165080
Selected:       1000
      mass1              mass2        ...         spin2z        
------------------ ------------------ ... ----------------------
1.6143461935827454 1.3480082834684102 ...    0.15321508762300204
  8.53730568294409  8.062443514085881 ...    0.17290926586270836
2.3332455377324335 2.1324722085786725 ... 0.00037066498754133016
1.7687002147784323 1.4471073453060435 ...   -0.12870575424207895


In [85]:
gwcosmo = GWCosmo(cosmology.Planck15)

max_z = gwcosmo.get_max_z(
    psds,
    waveform="o2-uberbank",  # auto-selects TaylorF2 or SEOBNRv4_ROM per event
    f_low=25.0,
    snr_threshold=1.0,
    min_triggers=1,
    mass1=np.array(test_samples["mass1"]),
    mass2=np.array(test_samples["mass2"]),
    spin1z=np.array(test_samples["spin1z"]),
    spin2z=np.array(test_samples["spin2z"]),
    jobs=1,
)

max_distance = gwcosmo.sensitive_distance(max_z).to_value(units.Mpc)

for i in range(len(test_samples[:2])):
    total_mass = test_samples["mass1"][i] + test_samples["mass2"][i]
    expected = "SEOBNRv4_ROM" if total_mass >= 4 else "TaylorF2threePointFivePN"
    print(f"\nEvent {i + 1}:")
    print(f"  mass1     = {test_samples['mass1'][i]:.2f} Msun")
    print(f"  mass2     = {test_samples['mass2'][i]:.2f} Msun")
    print(f"  total     = {total_mass:.2f} Msun  =>  {expected}")
    print(f"  spin1z    = {test_samples['spin1z'][i]:.3f}")
    print(f"  spin2z    = {test_samples['spin2z'][i]:.3f}")
    print(f"  max_z     = {max_z[i]:.4f}")
    print(f"  max_dist  = {max_distance[i]:.1f} Mpc")

  0%|          | 0/1000 [00:00<?, ?it/s]


Event 1:
  mass1     = 1.61 Msun
  mass2     = 1.35 Msun
  total     = 2.96 Msun  =>  TaylorF2threePointFivePN
  spin1z    = 0.004
  spin2z    = 0.153
  max_z     = 0.7977
  max_dist  = 2472.2 Mpc

Event 2:
  mass1     = 8.54 Msun
  mass2     = 8.06 Msun
  total     = 16.60 Msun  =>  SEOBNRv4_ROM
  spin1z    = 0.364
  spin2z    = 0.173
  max_z     = 5.6847
  max_dist  = 5441.2 Mpc


##  Compute sensitive volume and weights

In [86]:
# Calculate V * T for each sample
probs = 1 / len(max_distance)
probs *= 4 / 3 * np.pi * max_distance**3
volume = probs.sum()
probs /= volume

In [87]:
# Volume weight for each sample: proportional to max_distance^3
probs = (4 / 3) * np.pi * max_distance**3 / len(max_distance)
volume = probs.sum()
probs /= volume  # normalize to get sampling weights

print(f"Total sensitive volume: {volume:.2f} Mpc^3")
print(f"Volumetric rate:        {len(test_samples) / volume:.4e} yr^-1 Mpc^-3")

Total sensitive volume: 148513429778.02 Mpc^3
Volumetric rate:        6.7334e-09 yr^-1 Mpc^-3


## Draw weighted injections

In [88]:
nsamples = 400

dist_rv = stats.rv_discrete(values=(np.arange(len(probs)), probs))
indices = dist_rv.rvs(size=nsamples)

# Intrinsic parameters from weighted samples
cols = {
    key: np.array(test_samples[key])[indices]
    for key in ["mass1", "mass2", "spin1z", "spin2z"]
}

print(f"Drew {nsamples} injections")
print(f"mass1 range: [{cols['mass1'].min():.2f}, {cols['mass1'].max():.2f}] Msun")
print(f"mass2 range: [{cols['mass2'].min():.2f}, {cols['mass2'].max():.2f}] Msun")

Drew 400 injections
mass1 range: [1.20, 42.55] Msun
mass2 range: [1.09, 30.36] Msun


## Draw extrinsic parameters (sky position, distance, orientation)


In [89]:
# Sky position: uniform on the sphere
cols["longitude"] = stats.uniform(0, 2 * np.pi).rvs(size=nsamples)
cols["latitude"] = np.arcsin(stats.uniform(-1, 2).rvs(size=nsamples))

# Orientation
cols["inclination"] = np.arccos(stats.uniform(-1, 2).rvs(size=nsamples))
cols["polarization"] = stats.uniform(0, 2 * np.pi).rvs(size=nsamples)
cols["coa_phase"] = stats.uniform(-np.pi, 2 * np.pi).rvs(size=nsamples)

# Coalescence time: uniform over one year
cols["time_geocent"] = stats.uniform(1e9, units.year.to(units.second)).rvs(
    size=nsamples
)

print("Extrinsic parameters drawn:")
# print(f"  distance range: [{cols['distance'].min():.1f}, {cols['distance'].max():.1f}] Mpc")
print(
    f"  longitude range: [{cols['longitude'].min():.3f}, {cols['longitude'].max():.3f}] rad"
)

Extrinsic parameters drawn:
  longitude range: [0.002, 6.271] rad


## Convert from sensitive distance to redshift and comoving distance

In [90]:
# Convert from sensitive distance to redshift and comoving distance.
# FIXME: Replace this brute-force lookup table with a solver. (From ligo.skymap note)

# Distance drawn from p(d) ∝ d^2 up to max_distance for each event
cols["distance"] = stats.powerlaw(a=3, scale=max_distance[indices]).rvs(size=nsamples)

z_grid = np.linspace(0, max_z.max(), 10000)

ds = units.Quantity(
    list(progress_map(gwcosmo.sensitive_distance, z_grid, jobs=1))
).to_value(units.Mpc)

dc = units.Quantity(
    list(progress_map(gwcosmo.cosmo.comoving_distance, z_grid, jobs=1))
).to_value(units.Mpc)

z_for_ds = interp1d(ds, z_grid, kind="cubic", assume_sorted=True)
dc_for_ds = interp1d(ds, dc, kind="cubic", assume_sorted=True)

# Convert sensitive distance => comoving distance
zp1 = 1 + z_for_ds(cols["distance"])
cols["distance"] = dc_for_ds(cols["distance"])

# Apply redshift: source frame => observer frame
for key in ["distance", "mass1", "mass2"]:
    cols[key] *= zp1

print("After redshift correction (observer frame):")
print(f"  distance: [{cols['distance'].min():.1f}, {cols['distance'].max():.1f}] Mpc")
print(f"  mass1:    [{cols['mass1'].min():.2f}, {cols['mass1'].max():.2f}] Msun")
print(f"  mass2:    [{cols['mass2'].min():.2f}, {cols['mass2'].max():.2f}] Msun")

  0%|          | 0/10000 [00:00<?, ?it/s]

  0%|          | 0/10000 [00:00<?, ?it/s]

After redshift correction (observer frame):
  distance: [278.7, 90363.8] Mpc
  mass1:    [1.62, 326.11] Msun
  mass2:    [1.27, 219.21] Msun


## Write the injections to XML file

In [91]:
os.makedirs("runs/O4HL/fullpop4/", exist_ok=True)

# Build the XML document
xmldoc = ligolw.Document()
xmlroot = xmldoc.appendChild(ligolw.LIGO_LW())


# Register process exactly like the source code does:
process = ligolw_process.register_to_xmldoc(
    xmldoc,
    program="bayestar-inject",
    paramdict={},
)

# Populate sim_inspiral table
sims = xmlroot.appendChild(lsctables.SimInspiralTable.new())
for row in zip(*cols.values()):
    sims.appendRow(
        **dict(
            dict.fromkeys(sims.validcolumns, None),
            process_id=process.process_id,
            simulation_id=sims.get_next_id(),
            waveform="o2-uberbank",
            f_lower=25.0,
            **dict(zip(cols.keys(), row)),
        )
    )

# Record process end time
process.set_end_time_now()

print(f"Number of injections: {len(sims)}")

# Write to XML file
output_path = "runs/O4HL/fullpop4/injections.xml"
with open(output_path, "wb") as f:
    write_fileobj(xmldoc, f)

size = os.path.getsize(output_path) / 1e6
print(f"   Saved: {output_path} ({size:.2f} MB)")
print(f"   Number of injections: {len(sims)}")

Number of injections: 400
   Saved: runs/O4HL/fullpop4/injections.xml (0.09 MB)
   Number of injections: 400


## Load PSDs

In [92]:
# Load the reference PSDs from the XML file.
# These describe the noise level of each detector and are used
# to compute the expected SNR for each simulated signal.
xmldoc_psd = ligolw_utils.load_fileobj(
    open("runs/O4HL/psds.xml", "rb"), contenthandler=lal.series.PSDContentHandler
)
psds_raw = lal.series.read_psd_xmldoc(xmldoc_psd, root_name=None)

# Interpolate the PSDs so they can be evaluated at arbitrary frequencies.
psds = {
    key: bayestar_filter.InterpolatedPSD(bayestar_filter.abscissa(psd), psd.data.data)
    for key, psd in psds_raw.items()
    if psd is not None
}

# O4HL network: LIGO Hanford (H1) and LIGO Livingston (L1)
detectors_names = ["H1", "L1"]
psds_list = [psds[ifo] for ifo in detectors_names]

print(f"PSDs loaded: {list(psds.keys())}")

PSDs loaded: ['H1', 'L1']


## Load injections

In [93]:
# Load the simulated GW signals produced by bayestar-inject.
# Each row in the SimInspiral table represents one simulated binary merger
# with its intrinsic parameters (masses, spins) and extrinsic parameters
# (sky position, distance, orientation, coalescence time).
inj_xmldoc = ligolw_utils.load_filename(
    "runs/O4HL/fullpop4/injections.xml", contenthandler=ContentHandler
)
sim_inspiral_table = SimInspiralTable.get_table(inj_xmldoc)

print(f"Total injections: {len(sim_inspiral_table)}")
print("First injection:")
row = sim_inspiral_table[0]
print(f"  mass1    = {row.mass1:.2f} Msun")
print(f"  mass2    = {row.mass2:.2f} Msun")
print(f"  distance = {row.distance:.1f} Mpc")

Total injections: 400
First injection:
  mass1    = 3.59 Msun
  mass2    = 3.16 Msun
  distance = 3864.0 Mpc


## Prepare detector geometry

In [ ]:
# For each detector, we need its antenna response (how sensitive it is
# to signals from different sky directions) and its location on Earth
# (used to compute the time delay between detectors).
detectors = [lalsimulation.DetectorPrefixToLALDetector(ifo) for ifo in detectors_names]
responses = [det.response for det in detectors]
locations = [det.location for det in detectors]

for ifo, det, response, location in zip(
    detectors_names, detectors, responses, locations
):
    print(f"\n{'=' * 50}")
    print(f"Detector: {ifo} ({det.frDetector.name})")

    # Location: Cartesian (x, y, z) coordinates of the detector on Earth.
    # Used to compute the time delay of the signal between detectors:
    #   toa = lal.TimeDelayFromEarthCenter(location, ra, dec, epoch)
    # Note: ra and dec come from sim_inspiral.longitude and sim_inspiral.latitude
    # (LIGO-LW historical naming convention, see lsctables.py SimInspiral class)
    print("\nLocation on Earth (meters):")
    print(f"  x = {location[0]:>15.1f} m")
    print(f"  y = {location[1]:>15.1f} m")
    print(f"  z = {location[2]:>15.1f} m")
    print(f"  |r| = {np.sqrt(np.sum(location**2)):.1f} m  (≈ Earth radius)")

    # Response: 3x3 detector tensor encoding the orientation of the two arms.
    # Used to compute the antenna pattern functions F+ and Fx:
    #   Fplus, Fcross = lal.ComputeDetAMResponse(response, ra, dec, psi, gmst)
    # where psi=polarization angle and gmst=Greenwich Mean Sidereal Time at merger
    print("\nResponse matrix (detector tensor d_ij):")
    for row in response:
        print(f"  [{row[0]:>8.5f}  {row[1]:>8.5f}  {row[2]:>8.5f}]")

    # Antenna pattern for a concrete example (ra=0, dec=0, psi=0)
    Fplus, Fcross = lal.ComputeDetAMResponse(
        response,
        0.0,
        0.0,
        0.0,
        lal.GreenwichMeanSiderealTime(lal.LIGOTimeGPS(1000000000)),
    )
    print("\nAntenna pattern example (ra=0, dec=0, psi=0):")
    print(f"  F+ = {Fplus:.4f}  (sensitivity to + polarization)")
    print(f"  Fx = {Fcross:.4f}  (sensitivity to x polarization)")

Detectors:  ['H1', 'L1']
Responses:  [(3, 3), (3, 3)]
Locations:  [(3,), (3,)]


## Verify the download

In [95]:
# For each injection, simulate_snr() computes:
#   - horizon:  the distance at which this signal would have SNR=1
#   - snr:      the measured SNR in this detector
#   - phase:    the measured phase at coalescence
#   - toa:      the measured time of arrival
#   - series:   the full complex SNR time series (used later by BAYESTAR)
#
# The measurement_error="gaussian-noise" option adds realistic Gaussian
# noise to the SNR measurements, mimicking a real matched filter output.

sim = sim_inspiral_table[0]

simulation = simulate(
    seed=42,
    sim_inspiral=sim,
    psds=psds_list,
    responses=responses,
    locations=locations,
    measurement_error="gaussian-noise",
    f_low=None,  # use value from sim_inspiral table
    f_high=None,  # no high frequency cutoff
    waveform=None,  # use value from sim_inspiral table (o2-uberbank)
)

print(
    f"Simulated SNRs for event 0 (m1={sim.mass1:.1f}, m2={sim.mass2:.1f} Msun, d={sim.distance:.0f} Mpc):"
)
for ifo, (horizon, snr, phase, toa, series) in zip(detectors_names, simulation):
    print(f"  {ifo}:  horizon={horizon:.1f} Mpc  SNR={snr:.2f}  phase={phase:.3f} rad")

Simulated SNRs for event 0 (m1=3.6, m2=3.2 Msun, d=3864 Mpc):
  H1:  horizon=5630.8 Mpc  SNR=2.86  phase=-1.987 rad
  L1:  horizon=5987.7 Mpc  SNR=2.67  phase=-3.032 rad


## Apply detection thresholds

In [96]:
# A signal is considered detected if ALL of the following conditions are met:
#   1. The detector is online (duty_cycle=0.7 means 70% uptime)
#   2. At least min_triggers detectors see SNR >= snr_threshold (=1)
#   3. The network SNR (quadrature sum) >= net_snr_threshold (=8)
#
# These thresholds mimic the standard Observing scenarios criteria

snr_threshold = 1.0  # single-detector SNR threshold
net_snr_threshold = 8.0  # network SNR threshold (standard detection criterion)
duty_cycle = 0.7  # 70% detector uptime (realistic for O4)
min_triggers = 1  # minimum number of detectors that must fire

net_snr = 0.0
count_triggers = 0

for ifo, (horizon, snr, phase, toa, series) in zip(detectors_names, simulation):
    # Step 1: check if detector is online
    if np.random.uniform() > duty_cycle:
        print(f"  {ifo}: offline (duty cycle)")
        continue

    # Step 2: check single-detector SNR threshold
    if snr >= snr_threshold:
        count_triggers += 1
        net_snr += np.square(snr)  # accumulate SNR^2 for network SNR
        print(f"  {ifo}: SNR={snr:.2f} above threshold")
    else:
        print(f"  {ifo}: SNR={snr:.2f} below threshold")

# Step 3: compute network SNR and apply threshold
net_snr = np.sqrt(net_snr)
print(f"\nNetwork SNR:    {net_snr:.2f}  (threshold={net_snr_threshold})")
print(f"Triggers found: {count_triggers} (minimum={min_triggers})")

if count_triggers >= min_triggers and net_snr >= net_snr_threshold:
    print("\n Event DETECTED")
else:
    print("\n Event NOT detected")

  H1: SNR=2.86 above threshold
  L1: SNR=2.67 above threshold

Network SNR:    3.91  (threshold=8.0)
Triggers found: 2 (minimum=1)

 Event NOT detected


## Loop over all events

## Build output XML document

In [97]:
# Build output XML document
xmldoc_out = ligolw.Document()
xmlroot_out = xmldoc_out.appendChild(ligolw.LIGO_LW())

# Create all required tables
coinc_def_table = xmlroot_out.appendChild(CoincDefTable.new())
coinc_inspiral_table = xmlroot_out.appendChild(CoincInspiralTable.new())
coinc_map_table = xmlroot_out.appendChild(CoincMapTable.new())
coinc_table = xmlroot_out.appendChild(CoincTable.new())
xmlroot_out.appendChild(ProcessParamsTable.get_table(inj_xmldoc))
xmlroot_out.appendChild(ProcessTable.get_table(inj_xmldoc))
sim_inspiral_table_out = xmlroot_out.appendChild(SimInspiralTable.new())
sngl_inspiral_table_out = xmlroot_out.appendChild(SnglInspiralTable.new())
time_slide_table = xmlroot_out.appendChild(TimeSlideTable.new())

# Register process
process = ligolw_utils.process.register_to_xmldoc(
    xmldoc_out,
    program="bayestar-realize-coincs",
    paramdict={},
    comment="Simulated coincidences",
)

# Add search summary
all_time = segments.segment([lal.LIGOTimeGPS(0), lal.LIGOTimeGPS(2e9)])
append_search_summary(xmldoc_out, process, inseg=all_time, outseg=all_time)

# Create time slide entry
time_slide_id = time_slide_table.get_time_slide_id(
    {ifo: 0 for ifo in detectors_names}, create_new=process
)

# Populate CoincDef table
inspiral_coinc_def = copy.copy(InspiralCoincDef)
inspiral_coinc_def.coinc_def_id = coinc_def_table.get_next_id()
coinc_def_table.append(inspiral_coinc_def)

found_coinc_def = copy.copy(InspiralSCExactCoincDef)
found_coinc_def.coinc_def_id = coinc_def_table.get_next_id()
coinc_def_table.append(found_coinc_def)

print("Output XML structure ready")

Output XML structure ready


## Detection loop + storage

In [98]:
detected = []
missed = []

for i, sim in enumerate(sim_inspiral_table):
    simulation = simulate(
        seed=i + 1,
        sim_inspiral=sim,
        psds=psds_list,
        responses=responses,
        locations=locations,
        measurement_error="gaussian-noise",
    )

    net_snr = 0.0
    count_triggers = 0
    sngl_inspirals = []
    used_snr_series = []

    for ifo, (horizon, abs_snr, arg_snr, toa, series) in zip(
        detectors_names, simulation
    ):
        if np.random.uniform() > duty_cycle:
            continue
        if abs_snr >= snr_threshold:
            count_triggers += 1
            net_snr += np.square(abs_snr)
        used_snr_series.append(series)
        sngl_inspirals.append(
            sngl_inspiral_table_out.RowType(
                **dict(
                    dict.fromkeys(sngl_inspiral_table_out.validcolumns, None),
                    process_id=process.process_id,
                    ifo=ifo,
                    mass1=sim.mass1,
                    mass2=sim.mass2,
                    spin1z=sim.spin1z,
                    spin2z=sim.spin2z,
                    end=toa,
                    snr=abs_snr,
                    coa_phase=arg_snr,
                    eff_distance=horizon / abs_snr,
                )
            )
        )

    net_snr = np.sqrt(net_snr)

    if count_triggers < min_triggers or net_snr < net_snr_threshold:
        missed.append(i)
        # print(f"Event {i:3d}: m1={sim.mass1:5.1f} m2={sim.mass2:5.1f} Msun  "
        #       f"d={sim.distance:7.0f} Mpc  net_snr={net_snr:.2f} ==> NOT detected")
        continue

    detected.append(i)
    print(
        f"Event {i:3d}: m1={sim.mass1:5.1f} m2={sim.mass2:5.1f} Msun  "
        f"d={sim.distance:7.0f} Mpc  net_snr={net_snr:.2f} ==> Detected"
    )

    # Store Coinc
    coinc = coinc_table.appendRow(
        coinc_event_id=coinc_table.get_next_id(),
        process_id=process.process_id,
        coinc_def_id=inspiral_coinc_def.coinc_def_id,
        time_slide_id=time_slide_id,
        insts=detectors_names,
        nevents=len(detectors_names),
        likelihood=None,
    )

    # Store CoincInspiral
    coinc_inspiral_table.appendRow(
        coinc_event_id=coinc.coinc_event_id,
        instruments=[s.ifo for s in sngl_inspirals],
        end=lal.LIGOTimeGPS(
            1e-9 * np.mean([s.end.ns() for s in sngl_inspirals if s.end is not None])
        ),
        mass=sim.mass1 + sim.mass2,
        mchirp=sim.mchirp,
        combined_far=0.0,
        false_alarm_rate=0.0,
        minimum_duration=None,
        snr=net_snr,
    )

    # Store SnglInspiral + SNR series + CoincMap
    for sngl, series in zip(sngl_inspirals, used_snr_series):
        sngl.event_id = sngl_inspiral_table_out.get_next_id()
        sngl_inspiral_table_out.append(sngl)
        elem = lal.series.build_COMPLEX8TimeSeries(series, encoding="base64")
        elem.appendChild(Param.from_pyvalue("event_id", sngl.event_id))
        xmlroot_out.appendChild(elem)
        coinc_map_table.appendRow(
            coinc_event_id=coinc.coinc_event_id,
            table_name=sngl_inspiral_table_out.tableName,
            event_id=sngl.event_id,
        )

    # Store injection
    sim.simulation_id = sim_inspiral_table_out.get_next_id()
    sim_inspiral_table_out.append(sim)

print(f"\nDetected:   {len(detected)}/{len(sim_inspiral_table)}")
print(f"Missed:     {len(missed)}/{len(sim_inspiral_table)}")
print(f"Efficiency: {len(detected) / len(sim_inspiral_table) * 100:.1f}%")

Event  69: m1= 58.0 m2= 52.8 Msun  d=   6263 Mpc  net_snr=8.92 ==> Detected
Event 380: m1= 11.2 m2=  9.9 Msun  d=    279 Mpc  net_snr=23.96 ==> Detected

Detected:   2/400
Missed:     398/400
Efficiency: 0.5%


In [99]:
for i, sim in enumerate(list(sim_inspiral_table)[:5]):
    print(
        f"Event {i}: m1={sim.mass1:.1f} m2={sim.mass2:.1f} "
        f"d={sim.distance:.0f} Mpc "
        f"total_mass={sim.mass1 + sim.mass2:.1f} Msun"
    )

Event 0: m1=3.6 m2=3.2 d=3864 Mpc total_mass=6.7 Msun
Event 1: m1=20.6 m2=10.2 d=11873 Mpc total_mass=30.8 Msun
Event 2: m1=65.5 m2=2.6 d=4627 Mpc total_mass=68.1 Msun
Event 3: m1=44.8 m2=29.5 d=18150 Mpc total_mass=74.3 Msun
Event 4: m1=2.9 m2=2.7 d=4098 Mpc total_mass=5.6 Msun


## Write output file

In [100]:
process.set_end_time_now()

output_path = "runs/O4HL/fullpop4/events.xml.gz"
with open(output_path, "wb") as f:
    write_fileobj(xmldoc_out, f)

size = os.path.getsize(output_path) / 1e6
print(f"Saved: {output_path} ({size:.2f} MB)")
print(f"Detected events in file: {len(coinc_inspiral_table)}")

Saved: runs/O4HL/fullpop4/events.xml.gz (0.03 MB)
Detected events in file: 2
